In [1]:
# Cell 1: Import thư viện và nạp dữ liệu đã tiền xử lý
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Đọc lại file dữ liệu đã làm sạch từ bước 02
df = pd.read_csv('../data/processed/data_cleaned.csv')

# Khai báo đúng 4 đặc trưng đầu vào và biến mục tiêu
feature_cols = [
    'Heart Rate',
    'Physical Activity Level',
    'Sleep Duration',
    'Stress Level'
]
X = df[feature_cols]
y = df['Quality of Sleep']

print("Kích thước dữ liệu:", X.shape)

Kích thước dữ liệu: (374, 4)


In [ ]:
#Chia tập Train / Test và chuẩn hóa đặc trưng (Scaling)

# 1. Chia tập Train/Test với tỷ lệ 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Khởi tạo bộ chuẩn hóa StandardScaler
scaler = StandardScaler()

# Fit và Transform trên tập Train; chỉ Transform trên tập Test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Lưu bộ chuẩn hóa để Dashboard hoặc Edge sử dụng khi có dữ liệu mới
joblib.dump(scaler, '../models/scaler.pkl')
print('-> Đã lưu bộ chuẩn hóa vào: models/scaler.pkl')

In [ ]:
#Huấn luyện mô hình Random Forest Baseline
# Khởi tạo mô hình Random Forest Baseline
rf_model = RandomForestRegressor(
    n_estimators=100, max_depth=10, random_state=42
)

# Huấn luyện mô hình trên tập Train
rf_model.fit(X_train_scaled, y_train)

print('Huấn luyện mô hình hoàn tất!')

In [ ]:
#Đánh giá mô hình trên tập kiểm thử (Test Set)
# Dự đoán trên tập Test
y_pred = rf_model.predict(X_test_scaled)

# Tính toán các chỉ số đánh giá
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH RANDOM FOREST ---')
print(
    f'MAE (Sai số tuyệt đối trung bình): {mae:.2f} điểm (thang 0-100)'
)  # MAE càng nhỏ càng tốt
print(f'RMSE (Sai số căn phương trung bình): {rmse:.2f}')
print(f'R2 Score (Hệ số xác định): {r2:.4f}')  # R2 càng gần 1 càng tốt

# Hiển thị thử 5 kết quả so sánh giữa thực tế và dự đoán
comparison_df = pd.DataFrame(
    {'Điểm thực tế (Thật)': y_test.values[:5], 'Điểm dự đoán (AI)': y_pred[:5]}
)
comparison_df

In [ ]:
#Xuất file mô hình và đánh giá mức độ quan trọng của đặc trưng
import matplotlib.pyplot as plt

# 1. Lưu file mô hình vào thư mục models/
model_path = '../models/sleep_rf_model.pkl'
joblib.dump(rf_model, model_path)
print('-> Đã lưu mô hình thành công vào:', model_path)

# 2. Xem đặc trưng nào ảnh hưởng lớn nhất đến chất lượng giấc ngủ
importance = rf_model.feature_importances_
feat_importance = pd.Series(importance, index=feature_cols).sort_values(
    ascending=True
)

plt.figure(figsize=(8, 4))
feat_importance.plot(kind='barh', color='#008080')
plt.title('Mức độ ảnh hưởng của các đặc trưng đến điểm giấc ngủ')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()